# Performance Analysis: v_localrun_performance_check

This notebook evaluates win/loss performance from BigQuery using `win_loss_flag`, `chosen_action`, `selection_score`, and `total_investment`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.cloud import bigquery

PROJECT_ID = 'modelraja'
DATASET = 'MODELS_DATASET'
VIEW_NAME = 'v_localrun_performance_check'
TABLE_FQN = f"{PROJECT_ID}.{DATASET}.{VIEW_NAME}"

client = bigquery.Client(project=PROJECT_ID)
print('Reading from:', TABLE_FQN)

In [ ]:
query = f"SELECT * FROM `{TABLE_FQN}`"
df = client.query(query).to_dataframe(create_bqstorage_client=True)
print('Rows:', len(df))
print('Columns:', df.columns.tolist())
df.head()

In [ ]:
work = df.copy()

required = ['win_loss_flag', 'chosen_action', 'selection_score', 'total_investment', 'Stock_Snapshot_Date']
missing = [c for c in required if c not in work.columns]
if missing:
    raise ValueError(f'Missing required columns in view: {missing}')

work['Stock_Snapshot_Date'] = pd.to_datetime(work['Stock_Snapshot_Date'], errors='coerce')
work['selection_score'] = pd.to_numeric(work['selection_score'], errors='coerce')
work['total_investment'] = pd.to_numeric(work['total_investment'], errors='coerce')
work['win_loss_flag'] = pd.to_numeric(work['win_loss_flag'], errors='coerce')

work['investment_status'] = 'Not_Invested'
work.loc[work['win_loss_flag'] == 1, 'investment_status'] = 'Win'
work.loc[work['win_loss_flag'] == -1, 'investment_status'] = 'Loss'

invested = work[work['investment_status'].isin(['Win', 'Loss'])].copy()
not_invested = work[work['investment_status'] == 'Not_Invested'].copy()

invested['is_win'] = (invested['investment_status'] == 'Win').astype(int)
invested['is_loss'] = (invested['investment_status'] == 'Loss').astype(int)

print('Invested rows:', len(invested))
print('Not invested rows:', len(not_invested))
work['investment_status'].value_counts(dropna=False)


In [ ]:
total_rows = len(work)
wins = int((work['investment_status'] == 'Win').sum())
losses = int((work['investment_status'] == 'Loss').sum())
not_invested_count = int((work['investment_status'] == 'Not_Invested').sum())
invested_rows = wins + losses

overall = pd.DataFrame({
    'total_rows': [total_rows],
    'invested_rows': [invested_rows],
    'wins': [wins],
    'losses': [losses],
    'not_invested': [not_invested_count],
})
overall['invested_win_rate'] = overall['wins'] / overall['invested_rows'].replace({0: pd.NA})
overall['overall_hit_rate'] = overall['wins'] / overall['total_rows'].replace({0: pd.NA})
overall


In [ ]:
by_action = (
    invested.groupby('chosen_action', dropna=False)
    .agg(total=('is_win', 'count'), wins=('is_win', 'sum'), losses=('is_loss', 'sum'))
    .reset_index()
)
by_action['win_rate'] = by_action['wins'] / by_action['total']
by_action.sort_values('win_rate', ascending=False)


In [ ]:
daily = (
    invested.assign(date=invested['Stock_Snapshot_Date'].dt.date)
    .groupby('date', dropna=False)
    .agg(total=('is_win', 'count'), wins=('is_win', 'sum'))
    .reset_index()
)
daily['win_rate'] = daily['wins'] / daily['total']
display(daily.tail(20))

plt.figure(figsize=(10, 4))
plt.plot(daily['date'], daily['win_rate'], marker='o')
plt.title('Daily Invested Win Rate')
plt.xlabel('Date')
plt.ylabel('Win Rate')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Weighted performance by total_investment (invested rows only)
weighted = invested.copy()
weighted['w'] = weighted['total_investment'].fillna(0.0).clip(lower=0.0)
if weighted['w'].sum() == 0:
    weighted_win_rate = np.nan
else:
    weighted_win_rate = (weighted['is_win'] * weighted['w']).sum() / weighted['w'].sum()

pd.DataFrame({
    'weighted_invested_win_rate_by_total_investment': [weighted_win_rate],
    'total_weight': [weighted['w'].sum()]
})


In [ ]:
# Confidence bucket analysis using selection_score (invested only)
bucketed = invested.copy()
bucketed = bucketed[bucketed['selection_score'].notna()].copy()
bins = [0.0, 0.5, 0.6, 0.7, 0.8, 0.9, 1.01]
labels = ['<0.5', '0.5-0.6', '0.6-0.7', '0.7-0.8', '0.8-0.9', '0.9-1.0']
bucketed['score_bucket'] = pd.cut(bucketed['selection_score'], bins=bins, labels=labels, right=False)
bucket_summary = (
    bucketed.groupby('score_bucket', dropna=False)
    .agg(total=('is_win', 'count'), wins=('is_win', 'sum'))
    .reset_index()
)
bucket_summary['win_rate'] = bucket_summary['wins'] / bucket_summary['total']
bucket_summary


In [ ]:
# Not invested rows (win_loss_flag is NULL)
display(not_invested.head(20))
print('Not invested count:', len(not_invested))
